# Udyam Saathi — Voice Agent Pipeline Test (Colab)

Tests the **Tier 0 conversational layer** end-to-end without needing a mic or a deployed FastAPI backend:

```
mock transcript -> Groq intent router (tool-call) -> deterministic engine -> Groq response composer -> (optional) Bhashini/Sarvam TTS
```

**What this proves:** the LLM only ever *routes* and *phrases* — every rupee figure comes from the deterministic Python function, never from a chat completion. That's asserted explicitly in the last cell.

**Requirements:** a Groq API key is required. Bhashini/Sarvam keys are optional — TTS cells are skipped gracefully if absent.

In [1]:
!pip install -q groq requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 4.3 MB/s eta 0:00:00


In [8]:
from getpass import getpass
import os

os.environ["GROQ_API_KEY"] = getpass("Groq API key (required — intent routing + response phrasing): ")

# Not needed for this notebook — Section 7 uses free local Whisper + Meta MMS instead.
# Left here only if you later want to A/B test Bhashini/Sarvam against the free models.
os.environ["BHASHINI_API_KEY"] = getpass("Bhashini API key (optional, press Enter to skip): ")
os.environ["SARVAM_API_KEY"] = getpass("Sarvam API key (optional, press Enter to skip): ")

Groq API key (required — intent routing + response phrasing): ··········
Bhashini API key (optional, press Enter to skip): ··········
Sarvam API key (optional, press Enter to skip): ··········


## 1. Deterministic Tier 1 engine (simplified stand-in)

This mirrors `financial_calculator.py` and `government_schemes.py` closely enough to test the pipeline logic. Swap these for real imports once you run this against your actual repo (e.g. `!git clone` your repo into Colab and import directly instead).

In [9]:
def calculate_emi_schedule(amount: float, tenure_years: float = 5, interest_rate: float = 0.09, moratorium_months: int = 0, **kwargs):
    """Pure deterministic EMI calculator — the ONLY source of truth for EMI figures."""
    n_months = int(tenure_years * 12)
    r_m = interest_rate / 12
    n = n_months - moratorium_months
    if r_m == 0:
        emi = amount / n
    else:
        emi = amount * (r_m * (1 + r_m) ** n) / ((1 + r_m) ** n - 1)
    total_interest = (emi * n - amount) + (amount * r_m * moratorium_months)
    return {
        "loan_amount": amount,
        "monthly_emi": round(emi, 2),
        "tenure_years": tenure_years,
        "total_interest_payable": round(total_interest, 2),
        "moratorium_months": moratorium_months,
    }

def rank_eligible_schemes(investment_amount: float, business_category: str = "manufacturing", social_category: str = "general", **kwargs):
    """Simplified scheme registry lookup — pure lookup + arithmetic, no LLM."""
    schemes = [
        {"name": "PMEGP", "subsidy_pct": 0.25 if social_category != "general" else 0.15, "max_outlay": 5000000},
        {"name": "MUDRA Kishore", "subsidy_pct": 0.0, "max_outlay": 500000},
        {"name": "Stand-Up India", "subsidy_pct": 0.85 if social_category in ("sc", "st", "women") else 0.0, "max_outlay": 10000000},
    ]
    ranked = []
    for s in schemes:
        if investment_amount <= s["max_outlay"]:
            subsidy_amount = round(investment_amount * s["subsidy_pct"], 2)
            emi_info = calculate_emi_schedule(investment_amount * (1 - s["subsidy_pct"]))
            net_benefit = round(subsidy_amount - emi_info["total_interest_payable"], 2)
            ranked.append({"scheme": s["name"], "subsidy_amount": subsidy_amount, "net_benefit": net_benefit})
    ranked.sort(key=lambda x: x["net_benefit"], reverse=True)
    return {"ranked_schemes": ranked}

# sanity check
calculate_emi_schedule(500000, tenure_years=5)

{'loan_amount': 500000,
 'monthly_emi': 10379.18,
 'tenure_years': 5,
 'total_interest_payable': 122750.66,
 'moratorium_months': 0}

## 2. Intent router — real Groq call, tool-use mode

In [10]:
import json
from groq import Groq

client = Groq()

VOICE_TOOLS = [
    {"type": "function", "function": {
        "name": "calculate_emi",
        "description": "Compute EMI/DSCR for a given investment amount and tenure",
        "parameters": {"type": "object", "properties": {
            "amount": {"type": "number"},
            "tenure_years": {"type": "number"},
        }, "required": ["amount"]}}},
    {"type": "function", "function": {
        "name": "check_scheme_eligibility",
        "description": "Rank government schemes for the user's business profile",
        "parameters": {"type": "object", "properties": {
            "business_category": {"type": "string"},
            "social_category": {"type": "string"},
            "investment_amount": {"type": "number"},
        }, "required": ["investment_amount"]}}},
    {"type": "function", "function": {
        "name": "answer_general_question",
        "description": "Answer a conceptual/navigational question, or ask a clarifying question, WITHOUT stating any project-specific figures.",
        "parameters": {"type": "object", "properties": {"topic": {"type": "string"}}}}},
]

INTENT_ROUTER_SYSTEM_PROMPT = """You are the voice intent router for Udyam Saathi,
a rural enterprise feasibility platform. You are a dispatcher, not an advisor.

USER CONTEXT
- Detected language: {lang_code}
- Active project: {active_project_json}
- Recent conversation: {recent_turns}

HARD RULES
1. Never state a rupee amount, percentage, EMI, DSCR value, or viability verdict yourself.
   These come only from tool results, never from you.
2. Resolve ambiguous follow-ups (\"what about 8 lakh instead\") using active project + recent conversation.
3. Conceptual/navigation questions -> answer_general_question. Never state project-specific figures there.
4. Always respond with exactly one tool call, no extra commentary."""

def route_intent(transcript: str, lang_code: str = "en", active_project: dict | None = None, recent_turns: list | None = None):
    system_prompt = INTENT_ROUTER_SYSTEM_PROMPT.format(
        lang_code=lang_code,
        active_project_json=json.dumps(active_project or {}),
        recent_turns=json.dumps(recent_turns or []),
    )
    resp = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": transcript}],
        tools=VOICE_TOOLS,
        tool_choice="auto",
    )
    return resp.choices[0].message

## 3. Response composer — real Groq call, phrasing only

In [11]:
RESPONSE_COMPOSER_SYSTEM_PROMPT = """You are converting an already-computed result
from Udyam Saathi's deterministic engine into a short, warm, spoken answer in {lang_code}
for a rural entrepreneur.

COMPUTED RESULT (fixed facts — do not alter, recalculate, or round differently):
{tool_result_json}

RULES
1. State every number exactly as given. Never recompute, estimate, or round differently.
2. Never introduce a number not present above.
3. 2-4 short spoken sentences, no tables/bullets/symbols.
4. Simple, respectful {lang_code}, first-time-user friendly.
5. Never mention you are an AI or that numbers were \"calculated\" — state naturally."""

def compose_response(tool_result: dict, lang_code: str = "en"):
    system_prompt = RESPONSE_COMPOSER_SYSTEM_PROMPT.format(
        lang_code=lang_code, tool_result_json=json.dumps(tool_result)
    )
    resp = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": "Phrase this result."}],
    )
    return resp.choices[0].message.content

## 4. End-to-end orchestrator (mirrors `routers/voice.py`)

In [12]:
TOOL_DISPATCH = {
    "calculate_emi": calculate_emi_schedule,
    "check_scheme_eligibility": rank_eligible_schemes,
}

def run_pipeline(transcript: str, lang_code: str = "en", active_project: dict | None = None, recent_turns: list | None = None, verbose: bool = True):
    intent_msg = route_intent(transcript, lang_code, active_project, recent_turns)

    if not intent_msg.tool_calls:
        # answer_general_question / ambiguous case fell through without a tool call
        return {"transcript": transcript, "tool_used": None, "tool_result": None, "response_text": intent_msg.content}

    call = intent_msg.tool_calls[0]
    args = json.loads(call.function.arguments)

    if verbose:
        print(f"[router] transcript={transcript!r}")
        print(f"[router] tool={call.function.name} args={args}")

    if call.function.name == "answer_general_question":
        tool_result = {"note": "conceptual/navigation — no figures involved", "topic": args.get("topic")}
        response_text = compose_response(tool_result, lang_code)
    else:
        tool_fn = TOOL_DISPATCH[call.function.name]
        tool_result = tool_fn(**args)
        response_text = compose_response(tool_result, lang_code)

    return {
        "transcript": transcript,
        "tool_used": call.function.name,
        "tool_args": args,
        "tool_result": tool_result,
        "response_text": response_text,
    }

## 5. Run test transcripts

Simulated ASR output — replace with real Bhashini/Sarvam transcripts once you test with actual audio. Mix languages and follow-ups to stress-test context resolution.

In [13]:
session_turns = []  # running context for follow-up test

test_cases = [
    {"transcript": "What will my monthly EMI be if I invest 5 lakh rupees over 5 years?", "lang_code": "en"},
    {"transcript": "What about 8 lakh instead?", "lang_code": "en"},  # follow-up, tests context resolution
    {"transcript": "Which government schemes am I eligible for? I want to start a dairy business, investment is 3 lakh, I am from SC category.", "lang_code": "en"},
    {"transcript": "What does DSCR mean?", "lang_code": "en"},
]

results = []
for tc in test_cases:
    result = run_pipeline(tc["transcript"], tc["lang_code"], recent_turns=session_turns)
    results.append(result)
    session_turns.append({"transcript": result["transcript"], "response": result["response_text"]})
    print(f"\n--- Result ---\nTool: {result['tool_used']}\nSpoken answer: {result['response_text']}\n")

[router] transcript='What will my monthly EMI be if I invest 5 lakh rupees over 5 years?'
[router] tool=calculate_emi args={'amount': 500000, 'tenure_years': 5}

--- Result ---
Tool: calculate_emi
Spoken answer: You can borrow ₹500,000. Your monthly EMI will be ₹10,379.18 over a period of 5 years. The total interest you will pay is ₹122,750.66. There is no moratorium period.

[router] transcript='What about 8 lakh instead?'
[router] tool=calculate_emi args={'amount': 800000, 'tenure_years': 5}

--- Result ---
Tool: calculate_emi
Spoken answer: You can borrow Rs 800000.  
Your monthly EMi will be Rs 16606.68 for a 5‑year tenure.  
The total interest you’ll pay is Rs 196401.05.  
There is no moratorium period.

[router] transcript='Which government schemes am I eligible for? I want to start a dairy business, investment is 3 lakh, I am from SC category.'
[router] tool=check_scheme_eligibility args={'business_category': 'Dairy', 'investment_amount': 300000, 'social_category': 'SC'}

--- Re

## 6. Hallucination guard-rail check

Asserts the EMI figure spoken back in the composed response was NOT altered from the deterministic engine's own output — this is the exact check your `tests/test_voice_router.py` should also run.

In [14]:
emi_result = results[0]
expected = calculate_emi_schedule(500000, tenure_years=5)
actual_tool_result = emi_result["tool_result"]

assert actual_tool_result["monthly_emi"] == expected["monthly_emi"], "Tool result diverged from direct deterministic call!"
print(f"PASS: EMI figure is identical to the deterministic engine's own output -> {expected['monthly_emi']}")
print(f"Spoken phrasing (not authoritative, just presentation): {emi_result['response_text']}")
print("\nManually eyeball the spoken text above — confirm the number ₹", expected["monthly_emi"], "appears unchanged in it.")

PASS: EMI figure is identical to the deterministic engine's own output -> 10379.18
Spoken phrasing (not authoritative, just presentation): You can borrow ₹500,000. Your monthly EMI will be ₹10,379.18 over a period of 5 years. The total interest you will pay is ₹122,750.66. There is no moratorium period.

Manually eyeball the spoken text above — confirm the number ₹ 10379.18 appears unchanged in it.


## 7. Free ASR + TTS — Whisper + Meta MMS (runs locally, no API key)

- **ASR**: `faster-whisper` (open-source Whisper, CPU/GPU) — strong Hindi/Indian-language recognition, zero cost, zero key.
- **TTS**: Meta's **MMS-TTS** models via `transformers` — one small model per language, covers all 6 of your target languages, zero cost, zero key.

First run downloads model weights (a few hundred MB) — that's a one-time cost per Colab session, not a per-request one. Use Runtime -> Change runtime type -> GPU for faster inference (works on CPU too, just slower).

In [15]:
!pip install -q faster-whisper transformers scipy accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.5/39.5 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 73.3 MB/s eta 0:00:00


In [16]:
import torch
from faster_whisper import WhisperModel
from transformers import VitsModel, AutoTokenizer
import scipy.io.wavfile
from IPython.display import Audio, display
import os

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
COMPUTE_TYPE = "float16" if DEVICE == "cuda" else "int8"

# "small" is a good speed/accuracy tradeoff for a demo; use "medium" or "large-v3"
# for noticeably better Indian-language accuracy if you have GPU time to spare.
print("Loading Whisper (ASR)...")
whisper_model = WhisperModel("small", device=DEVICE, compute_type=COMPUTE_TYPE)

def transcribe_audio_free(filepath: str, language_hint: str | None = None):
    """language_hint: ISO code like 'hi', 'ta', 'mr' — leave None to auto-detect."""
    segments, info = whisper_model.transcribe(filepath, language=language_hint)
    text = " ".join(seg.text for seg in segments).strip()
    return text, info.language  # info.language is Whisper's detected ISO code

# Maps your 6 supported languages -> MMS-TTS model IDs (ISO 639-3-ish codes Meta uses)
MMS_LANG_MODELS = {
    "en": "facebook/mms-tts-eng",
    "hi": "facebook/mms-tts-hin",
    "mr": "facebook/mms-tts-mar",
    "ta": "facebook/mms-tts-tam",
    "te": "facebook/mms-tts-tel",
    "kn": "facebook/mms-tts-kan",
}
_tts_cache = {}

def synthesize_speech_free(text: str, lang_code: str = "en", out_path: str = "response.wav"):
    lang_key = lang_code.split("-")[0].lower()  # "en-IN" -> "en"
    model_name = MMS_LANG_MODELS.get(lang_key, MMS_LANG_MODELS["en"])
    if model_name not in _tts_cache:
        print(f"Loading TTS model for '{lang_key}' ({model_name})...")
        tok = AutoTokenizer.from_pretrained(model_name)
        mdl = VitsModel.from_pretrained(model_name).to(DEVICE)
        _tts_cache[model_name] = (tok, mdl)
    tok, mdl = _tts_cache[model_name]
    inputs = tok(text, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        waveform = mdl(**inputs).waveform
    scipy.io.wavfile.write(out_path, rate=mdl.config.sampling_rate, data=waveform.squeeze().cpu().numpy())
    return out_path

print(f"Ready — running on {DEVICE}. No API key needed.")

Loading Whisper (ASR)...


Ready — running on cpu. No API key needed.


## 8. Mic recorder (Colab JS bridge)

Records raw browser audio via `MediaRecorder`, saves it, then re-encodes to 16kHz mono WAV with `ffmpeg` (already installed in Colab) since most STT APIs expect clean WAV, not the browser's native webm/ogg.

In [22]:
from google.colab import output
from IPython.display import Javascript, display
from base64 import b64decode
import os

# --- Permission check ---
PERMISSION_JS = """
async function checkMic() {
  try {
    const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
    stream.getTracks().forEach(t => t.stop());
    return "granted";
  } catch (err) {
    return "denied: " + err.message;
  }
}
checkMic()
"""

def check_mic_permission():
    result = output.eval_js(PERMISSION_JS)
    print(f"Mic permission: {result}")
    if not result.startswith("granted"):
        print("\nFix: click the 🔒 or camera icon in your browser's address bar")
        print("(next to the URL), find 'Microphone', and set it to Allow for")
        print("this Colab tab. Then re-run this cell.")
    return result.startswith("granted")

# --- Recording ---
RECORD_JS = """
const sleep = time => new Promise(resolve => setTimeout(resolve, time))
var record = time => new Promise(async (resolve, reject) => {
  try {
    stream = await navigator.mediaDevices.getUserMedia({ audio: true })
    recorder = new MediaRecorder(stream)
    chunks = []
    recorder.ondataavailable = e => chunks.push(e.data)
    recorder.start()
    await sleep(time)
    recorder.onstop = async () => {
      blob = new Blob(chunks)
      text = await new Promise(res => {
        reader = new FileReader()
        reader.onloadend = () => res(reader.result)
        reader.readAsDataURL(blob)
      })
      resolve(text)
    }
    recorder.stop()
  } catch (err) {
    reject(err.message)
  }
})
"""

def record_audio(seconds: int = 5, raw_filename: str = "mic_raw.webm", wav_filename: str = "mic_input.wav"):
    display(Javascript(RECORD_JS))
    print(f"Recording for {seconds}s — speak now...")
    try:
        s = output.eval_js(f"record({seconds * 1000})", timeout_sec=seconds + 15)
    except Exception as e:
        print(f"Recording failed: {e}")
        print("Run check_mic_permission() and confirm it prints 'granted'.")
        return None
    b = b64decode(s.split(",")[1])
    with open(raw_filename, "wb") as f:
        f.write(b)
    os.system(f"ffmpeg -y -i {raw_filename} -ar 16000 -ac 1 {wav_filename} -loglevel quiet")
    print(f"Saved -> {wav_filename}")
    return wav_filename

# --- Run the check ---
check_mic_permission()

Mic permission: granted


True

## 9. Live voice loop — talk to it, it talks back (100% free)

Run the cell below, speak when prompted, and it will play the spoken answer back — no API key, no billing account, everything runs inside this Colab session. Run it again (same `live_session_turns` list) to have a follow-up conversation with context carried over.

Whisper auto-detects the language you spoke, so you can genuinely switch between English/Hindi/Marathi/etc. between turns and it'll follow.

In [ ]:
_active_result = {}   # holds only the last computed numbers, not a transcript log

def ask():
    """Run one turn: listens, answers, speaks back — remembers only the
    last computed result (numbers), not the full conversation transcript."""
    global _active_result

    wav_path = record_audio(seconds=6)
    if wav_path is None:
        print("Recording failed — check mic permission.")
        return

    transcript, detected_lang = transcribe_audio_free(wav_path)
    if not transcript:
        print("Didn't catch anything — try again.")
        return

    print(f"You said ({detected_lang}): {transcript}")

    # Only the last result gets passed as context — no transcript history
    result = run_pipeline(transcript, detected_lang, active_project=_active_result, verbose=False)

    # Update the remembered context ONLY if a real calculation happened
    if result.get("tool_result") and result["tool_used"] != "answer_general_question":
        _active_result = result["tool_result"]

    print(f"Udyam Saathi: {result['response_text']}")

    audio_path = synthesize_speech_free(result["response_text"], lang_code=detected_lang)
    display(Audio(audio_path, autoplay=True))

# Run this cell each time you want to ask one question
ask()

<IPython.core.display.Javascript object>

Recording for 6s — speak now...
Saved -> mic_input.wav
Didn't catch anything — try again.


## Notes / next steps

- **The live loop (Section 9) is real audio in, real audio out, and fully free** — Whisper (ASR) and Meta MMS-TTS (TTS) both run locally inside this Colab session, no API key or billing account needed. Sections 1–6 above it use typed transcripts on purpose, to isolate and validate the routing/hallucination-guard logic separately from the audio path.
- First run of Section 8/9 will trigger a browser mic-permission prompt — allow it, then re-run the cell. First run of Section 7 will also take a minute to download model weights.
- If a test case unexpectedly returns `tool_used: None` with a clarifying question instead of a tool call, that's the router correctly refusing to guess — not a bug.
- To test against your **real** `financial_calculator.py` / `government_schemes.py` instead of the simplified stand-ins in Cell 1, clone your repo into Colab (`!git clone <your-repo-url>`) and replace the imports in Cell 1 accordingly — the rest of the pipeline is unchanged.
- Try speaking in Hindi, Marathi, Tamil, Telugu, or Kannada — Whisper auto-detects the language and MMS-TTS will reply in the matching voice, as long as it's one of the 6 in `MMS_LANG_MODELS`.
- **For your actual deployed app**, prefer **Bhashini** over these local models — it's the free government API (no local GPU/model-hosting needed on your server), fits your "sovereign Digital India infra" pitch angle better, and is what your Antigravity-implemented `voice_asr_client.py`/`voice_tts_client.py` already target. Whisper+MMS here are for cost-free *notebook testing*, not necessarily your production stack — running Whisper+MMS in production would mean hosting/serving those models yourself, which has its own (compute) cost even if the model weights are free.
- Once this live loop works reliably, the same four functions (`record_audio`, `transcribe_audio_free`, `run_pipeline`, `synthesize_speech_free`) map directly onto `VoiceAgentWidget.tsx` + `routers/voice.py` in your real app — swap in Bhashini calls there with identical function signatures.